# Notebook 22.1 — Evaluación y selección de modelos

Implementamos desde cero (solo numpy) los tres protocolos de evaluación del módulo.
Los nombres de variables siguen el pseudocódigo de `03_evaluacion.md`.

**Índice**
1. Setup: datos y modelo base  
2. Algoritmo 22.1 — Protocolo train/test  
3. Algoritmo 22.2 — Selección de hiperparámetros (train/val/test)  
4. Algoritmo 22.3 — Validación cruzada k-fold  
5. Demostración empírica: sesgo del error de entrenamiento  
6. Selección de hiperparámetro: U-curva en $\lambda$  
7. **Stretch** — Descomposición bias-varianza empírica

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.legendre import legvander

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(42)

# --- True function and data generation ---
def true_f(x):
    return np.sin(np.pi * x)

SIGMA = 0.3
SIGMA2 = SIGMA ** 2  # Bayes floor = 0.09

def make_data(n, rng):
    """Draw n i.i.d. pairs (x, y) from p_data."""
    x = rng.uniform(-1, 1, n)
    y = true_f(x) + rng.normal(0, SIGMA, n)
    return x, y

# --- Feature map: Legendre polynomial basis (numerically stable) ---
def features(x, degree):
    """Return Legendre Vandermonde matrix of shape (len(x), degree+1)."""
    return legvander(x, degree)

# --- Ridge regression (closed-form, 1/m normalized) ---
def fit_ridge(x_train, y_train, degree, lam):
    """w* = (X^T X / m + lam I)^{-1} X^T y / m"""
    X = features(x_train, degree)
    m = len(y_train)
    A = X.T @ X / m + lam * np.eye(degree + 1)
    b = X.T @ y_train / m
    return np.linalg.solve(A, b)

def predict(x, w, degree):
    return features(x, degree) @ w

def mse(y_hat, y_true):
    return float(np.mean((y_hat - y_true) ** 2))

# Dataset for all experiments
N_TOTAL = 100
x_all, y_all = make_data(N_TOTAL, rng)
DEGREE = 5  # fixed model degree for evaluation protocols
LAM = 0.001  # fixed regularization for protocol demos

print(f"Dataset: {N_TOTAL} points, degree={DEGREE}, lambda={LAM}")
print(f"Bayes error floor: sigma^2 = {SIGMA2}")

## 2. Algoritmo 22.1 — Protocolo train/test

```
Entrada: D de tamaño n, fracción α
1. Mezclar D
2. m_train ← ⌊α·n⌋
3. D_train ← D[1:m_train]
4. D_test  ← D[m_train+1:n]
5. θ̂ ← arg min R̂(θ; D_train)
6. ê_test ← R̂(θ̂; D_test)
7. return θ̂, ê_test
```

In [ ]:
def train_test_split(x, y, alpha, rng):
    """
    Algorithm 22.1: simple train/test split.

    Parameters
    ----------
    alpha : float  fraction for training

    Returns
    -------
    w_hat : fitted weights
    e_test : test MSE estimate
    """
    n = len(x)
    # Step 1: shuffle
    idx = rng.permutation(n)
    x, y = x[idx], y[idx]
    # Step 2
    m_train = int(alpha * n)
    # Steps 3-4
    D_train = (x[:m_train], y[:m_train])
    D_test  = (x[m_train:], y[m_train:])
    # Step 5: fit
    w_hat = fit_ridge(*D_train, DEGREE, LAM)
    # Step 6: evaluate
    y_hat = predict(D_test[0], w_hat, DEGREE)
    e_test = mse(y_hat, D_test[1])
    # Step 7
    return w_hat, e_test

w_hat, e_test = train_test_split(x_all, y_all, alpha=0.8, rng=rng)
print(f"Algoritmo 22.1 — Test MSE: {e_test:.4f}  (Bayes floor: {SIGMA2:.2f})")

## 3. Algoritmo 22.2 — Selección de hiperparámetros (train/val/test)

```
Entrada: D, fracciones α_tr, α_val, cuadrícula Λ
1. Mezclar D
2. D_train, D_val, D_test ← split(D, α_tr, α_val, ...)
3. Para cada λ ∈ Λ:
   a. θ̂_λ ← fit(D_train, λ)
   b. ê_val(λ) ← R̂(θ̂_λ; D_val)
4. λ* ← arg min ê_val(λ)
5. θ̂_final ← fit(D_train, λ*)
6. ê_test ← R̂(θ̂_final; D_test)
7. return θ̂_final, ê_test
```

In [ ]:
def train_val_test_split(x, y, alpha_tr, alpha_val, Lambda, rng):
    """
    Algorithm 22.2: hyperparameter selection with held-out validation set.

    Parameters
    ----------
    Lambda : array of lambda values to sweep

    Returns
    -------
    w_hat_final : fitted weights with lambda*
    lam_star    : selected hyperparameter
    e_test      : final test MSE (reported once)
    e_val_grid  : validation MSE for each lambda (for plotting)
    """
    n = len(x)
    # Step 1: shuffle
    idx = rng.permutation(n)
    x, y = x[idx], y[idx]
    # Step 2: three-way split
    m_train = int(alpha_tr * n)
    m_val   = int(alpha_val * n)
    D_train = (x[:m_train],              y[:m_train])
    D_val   = (x[m_train:m_train+m_val], y[m_train:m_train+m_val])
    D_test  = (x[m_train+m_val:],        y[m_train+m_val:])
    # Step 3: sweep lambda
    e_val_grid = np.zeros(len(Lambda))
    for k, lam in enumerate(Lambda):
        w_lam = fit_ridge(*D_train, DEGREE, lam)          # 3a
        y_val_hat = predict(D_val[0], w_lam, DEGREE)
        e_val_grid[k] = mse(y_val_hat, D_val[1])         # 3b
    # Step 4
    lam_star = Lambda[np.argmin(e_val_grid)]
    # Step 5
    w_hat_final = fit_ridge(*D_train, DEGREE, lam_star)
    # Step 6
    y_test_hat = predict(D_test[0], w_hat_final, DEGREE)
    e_test = mse(y_test_hat, D_test[1])
    # Step 7
    return w_hat_final, lam_star, e_test, e_val_grid, D_train, D_val, D_test

Lambda_grid = np.logspace(-5, 2, 60)
w_final, lam_star, e_test_alg2, e_val_grid, D_tr, D_va, D_te = train_val_test_split(
    x_all, y_all, alpha_tr=0.6, alpha_val=0.2, Lambda=Lambda_grid, rng=rng
)
print(f"Algoritmo 22.2 — λ* = {lam_star:.5f},  Test MSE = {e_test_alg2:.4f}")

## 4. Algoritmo 22.3 — Validación cruzada k-fold

```
Entrada: D de tamaño n, número de folds k
1. Mezclar D
2. Partir D en k bloques: D_1, ..., D_k
3. Para j = 1, ..., k:
   a. D_val^(j)   ← D_j
   b. D_train^(j) ← D \ D_j
   c. θ̂_j ← fit(D_train^(j))
   d. e_j ← R̂(θ̂_j; D_val^(j))
4. ê_CV ← (1/k) Σ e_j
5. return ê_CV
```

In [ ]:
def k_fold_cv(x, y, k, lam, rng):
    """
    Algorithm 22.3: k-fold cross-validation.

    Returns
    -------
    e_cv   : scalar CV estimate
    e_folds: array of per-fold errors (length k)
    """
    n = len(x)
    # Step 1: shuffle
    idx = rng.permutation(n)
    x, y = x[idx], y[idx]
    # Step 2: partition into k blocks
    fold_indices = np.array_split(np.arange(n), k)
    e_folds = np.zeros(k)
    # Step 3
    for j in range(k):
        val_idx   = fold_indices[j]                                  # 3a
        train_idx = np.concatenate([fold_indices[i]                  # 3b
                                    for i in range(k) if i != j])
        D_train_j = (x[train_idx], y[train_idx])
        D_val_j   = (x[val_idx],   y[val_idx])
        w_j = fit_ridge(*D_train_j, DEGREE, lam)                    # 3c
        y_hat_j = predict(D_val_j[0], w_j, DEGREE)
        e_folds[j] = mse(y_hat_j, D_val_j[1])                       # 3d
    # Step 4
    e_cv = np.mean(e_folds)
    return e_cv, e_folds

e_cv, e_folds = k_fold_cv(x_all, y_all, k=5, lam=LAM, rng=rng)
print(f"Algoritmo 22.3 — 5-fold CV MSE: {e_cv:.4f}")
print(f"  Errores por fold: {e_folds.round(4)}")

## 5. Demostración empírica: sesgo del error de entrenamiento

Repetimos B=200 particiones aleatorias train/test y comparamos las distribuciones de error.
El error de entrenamiento debe estar sistemáticamente desplazado hacia la izquierda.

In [ ]:
from scipy.stats import gaussian_kde

B = 200  # number of random splits
train_errors = np.zeros(B)
test_errors  = np.zeros(B)

for b in range(B):
    x_b, y_b = make_data(50, rng)
    idx = rng.permutation(50)
    m_train = 40
    x_tr, y_tr = x_b[idx[:m_train]], y_b[idx[:m_train]]
    x_te, y_te = x_b[idx[m_train:]], y_b[idx[m_train:]]
    w_b = fit_ridge(x_tr, y_tr, DEGREE, LAM)
    train_errors[b] = mse(predict(x_tr, w_b, DEGREE), y_tr)
    test_errors[b]  = mse(predict(x_te, w_b, DEGREE), y_te)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# KDE plot
ax = axes[0]
grid = np.linspace(0, max(test_errors) * 1.1, 300)
ax.fill_between(grid, gaussian_kde(train_errors)(grid), alpha=0.4, color='steelblue', label='Error train')
ax.fill_between(grid, gaussian_kde(test_errors)(grid),  alpha=0.4, color='tomato',   label='Error test')
ax.axvline(SIGMA2, color='k', lw=1.5, ls='--', label=f'$\\sigma^2={SIGMA2}$')
ax.set_xlabel('MSE')
ax.set_ylabel('Densidad')
ax.set_title('Distribución de errores train vs. test')
ax.legend()

# Scatter
ax = axes[1]
ax.scatter(train_errors, test_errors, alpha=0.4, s=20, color='steelblue')
lim = max(train_errors.max(), test_errors.max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', lw=1.5, label='y = x')
ax.set_xlabel('Error train')
ax.set_ylabel('Error test')
ax.set_title('Optimismo: casi todos los puntos están\npor encima de la diagonal')
ax.legend()

plt.suptitle(f'Sesgo del error de entrenamiento (B={B} experimentos)', y=1.02)
plt.tight_layout()
plt.show()

frac_above = np.mean(test_errors > train_errors)
print(f"Fracción de experimentos con error_test > error_train: {frac_above:.1%}")
print(f"Media train MSE: {train_errors.mean():.4f}")
print(f"Media test  MSE: {test_errors.mean():.4f}")

## 6. Selección de hiperparámetro: U-curva en $\lambda$

Barremos $\lambda$ en una cuadrícula, calculamos train/val/test MSE, y marcamos $\lambda^*$.

In [ ]:
# Generate fresh dataset
x_hyp, y_hyp = make_data(120, rng)
idx = rng.permutation(120)
x_hyp, y_hyp = x_hyp[idx], y_hyp[idx]
n_tr, n_val = 70, 25
x_tr_h, y_tr_h = x_hyp[:n_tr], y_hyp[:n_tr]
x_va_h, y_va_h = x_hyp[n_tr:n_tr+n_val], y_hyp[n_tr:n_tr+n_val]
x_te_h, y_te_h = x_hyp[n_tr+n_val:], y_hyp[n_tr+n_val:]

DEG_HYPER = 9  # moderate overfitting degree to see U-shape
lambdas = np.logspace(-5, 1, 80)
e_tr_curve  = np.zeros(len(lambdas))
e_val_curve = np.zeros(len(lambdas))
e_te_curve  = np.zeros(len(lambdas))

for k, lam in enumerate(lambdas):
    w_k = fit_ridge(x_tr_h, y_tr_h, DEG_HYPER, lam)
    e_tr_curve[k]  = mse(predict(x_tr_h, w_k, DEG_HYPER), y_tr_h)
    e_val_curve[k] = mse(predict(x_va_h, w_k, DEG_HYPER), y_va_h)
    e_te_curve[k]  = mse(predict(x_te_h, w_k, DEG_HYPER), y_te_h)

lam_star_idx = np.argmin(e_val_curve)
lam_star_h   = lambdas[lam_star_idx]

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(lambdas, e_tr_curve,  color='steelblue', label='Train MSE')
ax.semilogx(lambdas, e_val_curve, color='tomato',    label='Val MSE')
ax.semilogx(lambdas, e_te_curve,  color='seagreen',  label='Test MSE', ls='--')
ax.axvline(lam_star_h, color='k', ls=':', lw=1.5, label=f'$\\lambda^*={lam_star_h:.4f}$')
ax.axhline(SIGMA2, color='gray', ls='--', lw=1, label=f'$\\sigma^2={SIGMA2}$')
ax.set_xlabel('$\\lambda$ (escala log)')
ax.set_ylabel('MSE')
ax.set_title(f'U-curva de validación — grado {DEG_HYPER}')
ax.legend()
plt.tight_layout()
plt.show()

print(f"λ* = {lam_star_h:.5f}")
print(f"Val MSE en λ*: {e_val_curve[lam_star_idx]:.4f}")
print(f"Test MSE final (reportar solo este): {e_te_curve[lam_star_idx]:.4f}")

## 7. STRETCH — Descomposición bias-varianza empírica

Fijamos $m=30$, corremos $B=200$ conjuntos de entrenamiento bootstrap,
y computamos Bias², Var y MSE vs. grado polinomial.

Debe cumplirse: $\text{MSE} \approx \sigma^2 + \text{Bias}^2 + \text{Var}$ en cada grado.

In [ ]:
B_bv = 200
m_bv = 30
degrees_bv = np.arange(0, 13)

# Fixed test points
x_test_bv = np.linspace(-1, 1, 200)
f_star_test = true_f(x_test_bv)

bias2_list = []
var_list   = []
mse_list   = []

for deg in degrees_bv:
    # predictions[b, j] = prediction of model b at test point j
    predictions = np.zeros((B_bv, len(x_test_bv)))
    for b in range(B_bv):
        x_b, y_b = make_data(m_bv, rng)
        lam_bv = 1e-8 if deg <= 6 else 1e-6  # tiny regularization for stability
        w_b = fit_ridge(x_b, y_b, deg, lam_bv)
        predictions[b] = predict(x_test_bv, w_b, deg)

    mean_pred = predictions.mean(axis=0)              # E_D[f_hat(x)]
    bias2 = np.mean((mean_pred - f_star_test) ** 2)   # avg over test points
    var   = np.mean(predictions.var(axis=0))           # avg over test points
    mse_total = SIGMA2 + bias2 + var

    bias2_list.append(bias2)
    var_list.append(var)
    mse_list.append(mse_total)

bias2_arr = np.array(bias2_list)
var_arr   = np.array(var_list)
mse_arr   = np.array(mse_list)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(degrees_bv, bias2_arr, 'o-', color='tomato',    label='Bias²')
ax.plot(degrees_bv, var_arr,   's-', color='steelblue', label='Varianza')
ax.plot(degrees_bv, mse_arr,   '^-', color='mediumpurple', lw=2, label='MSE = σ² + Bias² + Var')
ax.axhline(SIGMA2, color='k', ls='--', lw=1.5, label=f'Piso Bayes $\\sigma^2={SIGMA2}$')
ax.set_xlabel('Grado del polinomio')
ax.set_ylabel('Error')
ax.set_title(f'Descomposición bias-varianza empírica (B={B_bv}, m={m_bv})')
ax.legend()
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

# Verify decomposition: MSE ≈ σ² + Bias² + Var
print("Verificación MSE = σ² + Bias² + Var:")
print(f"{'Grado':>6} {'Bias²':>8} {'Var':>8} {'σ²+B²+V':>10} {'MSE':>8} {'Error':>8}")
for i, deg in enumerate(degrees_bv):
    computed = SIGMA2 + bias2_arr[i] + var_arr[i]
    err = abs(computed - mse_arr[i])
    print(f"{deg:>6} {bias2_arr[i]:>8.4f} {var_arr[i]:>8.4f} {computed:>10.4f} {mse_arr[i]:>8.4f} {err:>8.2e}")